# This notebook runs the quantitative analysis
Please provide the right configuration and run the notebook through to the end

In [ ]:
import logging
import os
import sys
import torch
import yaml

from functools import partial
from logging.handlers import RotatingFileHandler
from jsonschema import validate, ValidationError

from esc_audio_dataset import ESCAudioDataset
from bcos.modules.bcosconv2d import BcosConv2d
from config.config_validation_template import CONFIG_TEMPLATE
from custom_logger_formatter import CustomLoggerFormatter
from gradcam import GradCAM, _get_gradcam_target_layer
from main import DATASET_MAPPING
from resnet_bcos import make_resnet18, make_resnet34, make_resnet50
from resnet_18_baseline import BaselineResNet
from tune import TUNABLE_PARAMS

import quant_analysis_utils as q_utils

### 0. Config & setup

In [ ]:
EXPERIMENT_DIR = r"output\13-06-2026--11-33" # change directory to the specific experiment

BEST_MODEL_DIR = r"job_0\best_model" # folder containing .pt model, relative from `EXPERIMENT_DIR`

DEVICE = "cuda" # use if CUDA or ROCm

In [ ]:
# Initialise Logger.
def _make_stream_handler(level: int) -> logging.StreamHandler:
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(level)
    ch.setFormatter(CustomLoggerFormatter())
    return ch

level: int=logging.DEBUG
logger = logging.getLogger("quant logger")
logger.setLevel(level)
logger.propagate = False  
ch = _make_stream_handler(level)
logger.addHandler(ch)
f_ch = RotatingFileHandler(f"{EXPERIMENT_DIR}/quantitative_analysis.log")
f_ch.setLevel(level)
f_ch.setFormatter(
    logging.Formatter(
        '%(asctime)s - %(name)s - %(levelname)s - %(message)s'
    )
)
logger.addHandler(f_ch)

logger.info(f"This file logs the quantitative analysis process.")

In [ ]:
# validate the provided config file.
with open(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/run_config.yml", 'r') as stream:
    CONFIG = yaml.safe_load(stream)

# Get the general config settings from the main yaml
with open(f"{EXPERIMENT_DIR}/config.yml", 'r') as stream:
    general_config = yaml.safe_load(stream)

CONFIG["general"] = general_config["general"]

try:
    validate(general_config, CONFIG_TEMPLATE)
except ValidationError as e:
    raise ValidationError(
        "\x1b[31;1mA validation error occurred in the config file" \
        f": {e.message}\x1b[0m"
    ) from e

if "jobs" in CONFIG.keys():
    CONFIG = CONFIG["jobs"]["job0"]
for tunable_param in TUNABLE_PARAMS.keys():
    if tunable_param == "small_inputs":
        CONFIG["model_params"][tunable_param] = CONFIG["model_params"][tunable_param][0]
    elif type(CONFIG[tunable_param]) == list:
        CONFIG[tunable_param] = CONFIG[tunable_param][0]

logger.info("config: {CONFIG}")

### 1. Load data


In [ ]:
dataset = ESCAudioDataset(
    data_dirs=DATASET_MAPPING["dirs"],
    folds=DATASET_MAPPING["train_folds"],
    csv_path=DATASET_MAPPING["csv_path"],
    target_sr=CONFIG["sample_rate"],
    duration=CONFIG["duration"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
    n_mels=CONFIG["n_mels"],
    top_db=CONFIG["top_db"],
)
logger.debug(f"Dataset size: {len(dataset)}")
logger.debug(f"Shape of first x element: {dataset[0][0].shape}")
logger.debug(f"First y element: {dataset[0][1]}")
test_data = ESCAudioDataset(
    data_dirs=DATASET_MAPPING["dirs"],
    folds=DATASET_MAPPING["test_folds"],
    csv_path=DATASET_MAPPING["csv_path"],
    target_sr=CONFIG["sample_rate"],
    duration=CONFIG["duration"],
    n_fft=CONFIG["n_fft"],
    hop_length=CONFIG["hop_length"],
    n_mels=CONFIG["n_mels"],
    top_db=CONFIG["top_db"],
)
logger.debug(f"Test dataset size: {len(test_data)}")

# Normalise
logger.debug("Fitting normalisation.")
dataset.fit_normalisation(list(range(len(dataset))))
test_data.mean = dataset.mean
test_data.std = dataset.std
logger.debug(
    "Normalisation fitted: "
    f"mean={dataset.mean}, std={dataset.std}"
)

### 2. Load model

In [ ]:
models = {
    "resnet18_bcos": (
        make_resnet18, {
            "logger": logger,
            "num_classes": dataset.get_n_classes(),
            "in_chans" : 1,
            "small_inputs": CONFIG["model_params"].get("small_inputs", True),
            "conv_layer": partial(BcosConv2d, b=CONFIG["b"], max_out=2),
        }
    ),
    "resnet34_bcos": (
        make_resnet34, {
            "logger": logger,
            "num_classes": dataset.get_n_classes(),
            "in_chans" : 1,
            "small_inputs": CONFIG["model_params"].get("small_inputs", True),
            "conv_layer": partial(BcosConv2d, b=CONFIG["b"], max_out=2),
        }
    ),
    "resnet50_bcos": (
        make_resnet50, {
            "logger": logger,
            "num_classes": dataset.get_n_classes(),
            "in_chans" : 1,
            "small_inputs": CONFIG["model_params"].get("small_inputs", True),
            "conv_layer": partial(BcosConv2d, b=CONFIG["b"], max_out=2),
        }
    ),
    "resnet18_baseline": (
        lambda **kwargs: BaselineResNet(kwargs["num_classes"], kwargs["logger"]),
        {"logger": logger, "num_classes": dataset.get_n_classes()}
    )   
}
model = None
for name, (cls, kwargs) in models.items():
    if CONFIG['model'].lower() in name:
        model = cls(**kwargs)
        break
assert model is not None, \
    f"Provided model in config does not exist ({model})."

logger.debug(f"Model:\n{model}")
logger.debug("Total number of parameters: "
    f"{sum(p.numel() for p in model.parameters()):,}"
)

model = model.to(DEVICE)


In [ ]:
model_file_path = [entry for entry in os.listdir(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}") if entry.endswith(".pth")][0]
logger.info(f"Loading model from {model_file_path}")

state_dict = torch.load(f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/{model_file_path}", weights_only=True)
model.load_state_dict(state_dict)

#### 2.1. Wrapping any non b-cos model with gradcam

In [ ]:
# explains final convolutional layer
if "baseline" in CONFIG['model'].lower(): 
    model = GradCAM(model, target_layer=_get_gradcam_target_layer(model))
    logger.info("Wrapped model in GradCam shell.")

### 3. Grid pointing game

In [ ]:
# train
train_grid_dir = f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/grid_train/"
os.makedirs(train_grid_dir, exist_ok=True)
train_grid_scores_absolute, train_grid_scores_weighted = q_utils.grid_pointing_game(
    dataset, 
    model, 
    DEVICE, 
    logger, 
    first_img_output_dir=train_grid_dir
)
logger.critical(f"Absolute grid pointing game results on train set: Total pairs evaluated: {len(train_grid_scores_absolute)} | Mean score: {train_grid_scores_absolute.mean() * 100:.5f}% | std: {train_grid_scores_absolute.std() * 100:.5f}%")
logger.critical(f"Relative grid pointing game results on train set: Total pairs evaluated: {len(train_grid_scores_weighted)} | Mean score: {train_grid_scores_weighted.mean() * 100:.5f}% | std: {train_grid_scores_weighted.std() * 100:.5f}%")

# test
test_grid_dir = f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/grid_test/"
os.makedirs(test_grid_dir, exist_ok=True)
test_grid_scores_absolute, test_grid_scores_weighted = q_utils.grid_pointing_game(
    test_data, 
    model, 
    DEVICE, 
    logger, 
    first_img_output_dir=test_grid_dir
)
logger.critical(f"Absolute grid pointing game results on test set: Total pairs evaluated: {len(test_grid_scores_absolute)} | Mean score: {test_grid_scores_absolute.mean() * 100:.5f}% | std: {test_grid_scores_absolute.std() * 100:.5f}%")
logger.critical(f"Relative grid pointing game results on test set: Total pairs evaluated: {len(test_grid_scores_weighted)} | Mean score: {test_grid_scores_weighted.mean() * 100:.5f}% | std: {test_grid_scores_weighted.std() * 100:.5f}%")

torch.cuda.empty_cache()

### 4. Positive/negative contribution mask accuracy
see how well the model predicts when only being able to use the pixels it used positively

In [ ]:
# train
train_mask_dir = f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/mask_train/"
os.makedirs(train_mask_dir, exist_ok=True)
original_train_accuracy, positive_masked_train_accuracy, negative_masked_train_accuracy = q_utils.contribution_masking_accuracy(
    dataset=dataset,
    model=model,
    DEVICE=DEVICE,
    logger=logger,
    first_img_output_dir=train_mask_dir
)

logger.info(f"Original train accuracy: {original_train_accuracy*100:.5f}")
logger.info(f"Positive masked train accuracy: {positive_masked_train_accuracy*100:.5f}")
logger.info(f"Negative masked train accuracy: {negative_masked_train_accuracy*100:.5f}")

# test
test_mask_dir = f"{EXPERIMENT_DIR}/{BEST_MODEL_DIR}/mask_test/"
os.makedirs(test_mask_dir, exist_ok=True)
original_test_accuracy, positive_masked_test_accuracy, negative_masked_test_accuracy = q_utils.contribution_masking_accuracy(
    dataset=test_data,
    model=model,
    DEVICE=DEVICE,
    logger=logger,
    first_img_output_dir=test_mask_dir
)

logger.info(f"Original test accuracy: {original_test_accuracy*100:.5f}")
logger.info(f"Positive masked test accuracy: {positive_masked_test_accuracy*100:.5f}")
logger.info(f"Negative masked test accuracy: {negative_masked_test_accuracy*100:.5f}")

torch.cuda.empty_cache()